# music2emo — Mood Prediction Benchmark

Evaluates **music2emo** (MERT embeddings + chord/key features, multi-task learning)
on three annotated datasets and five qualitative spot-checks.

> **Run this notebook in its own Colab runtime** — do not mix with `01_essentia_benchmark.ipynb`.
> music2emo requires numpy ≥2.x; Essentia requires numpy ~1.x.

### Sections
0. Environment setup
1. Shared evaluation utilities
2. Predictor
3. Datasets
4. Evaluation
5. Qualitative spot-checks
6. Profiling
7. Summary & cross-model comparison

## 0. Environment Setup

In [ ]:
import os

# Clone music2emo (not on PyPI)
if not os.path.exists("Music2Emotion"):
    print("Cloning music2emo from GitHub...")
    !git clone --depth 1 https://github.com/AMAAI-Lab/Music2Emotion.git
    # Strip torch== / torchaudio== pins: Colab's pre-installed torch is compatible
    # and the pinned 2.3.1 would downgrade it, breaking Colab's torchvision.
    with open("Music2Emotion/requirements.txt") as _f:
        _lines = [l for l in _f if not l.lower().startswith(("torch==", "torchaudio=="))]
    with open("/tmp/m2e_requirements.txt", "w") as _f:
        _f.writelines(_lines)
    !pip install -r /tmp/m2e_requirements.txt -q
    print("music2emo installed.")
else:
    print("Music2Emotion already present.")

!pip install yt-dlp matplotlib pandas scipy tqdm gdown -q
print("Setup complete.")

## 1. Shared Evaluation Utilities

In [ ]:
import sys, shutil

# REPO_BRANCH: update to "main" after the PR is merged
REPO_BRANCH = "feat/mood-model-benchmark"

if not os.path.exists("eval_datasets.py"):
    REPO = "Soundtrack-Mood-Manager"
    if os.path.exists(REPO) and not os.path.exists(f"{REPO}/evaluation"):
        print(f"Removing stale clone and re-cloning {REPO_BRANCH}...")
        shutil.rmtree(REPO)
    if not os.path.exists(REPO):
        !git clone --depth 1 --branch {REPO_BRANCH} https://github.com/francescovidaich964/Soundtrack-Mood-Manager.git
    sys.path.insert(0, f"{REPO}/evaluation")
sys.path.insert(0, "Music2Emotion")

from eval_datasets import setup_deam, setup_emomusic, setup_pmemo, setup_merge
from metrics import compute_metrics, print_metrics
from visualization import plot_scatter, cross_dataset_comparison
from spot_checks import SPOT_CHECKS, download_spot_checks, run_evaluation, profile_predictor

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR      = Path("data")
SPOTCHECK_DIR = Path("spotchecks")
for d in [DATA_DIR, SPOTCHECK_DIR]:
    d.mkdir(exist_ok=True)

MODEL_TAG = "music2emo"
print("Imports complete.")

## 2. Predictor

Downloads ~500 MB of weights from Hugging Face (`amaai-lab/music2emo`) on first instantiation.

In [ ]:
class Music2EmoPredictor:
    """music2emo: MERT embeddings + chord/key features, multi-task learning.

    Normalises output from the native 1–9 scale to [0, 1] to match Essentia.
    """

    def __init__(self):
        print("Loading music2emo (downloads weights on first run)...")
        from music2emo import Music2emo
        self._model = Music2emo()
        print("Music2EmoPredictor ready.")

    def predict(self, audio_path) -> dict | None:
        """Returns {'valence': float, 'arousal': float} in [0, 1], or None on failure."""
        try:
            out = self._model.predict(str(audio_path))
            valence = float(np.clip((out["valence"] - 1.0) / 8.0, 0.0, 1.0))
            arousal = float(np.clip((out["arousal"] - 1.0) / 8.0, 0.0, 1.0))
            return {"valence": valence, "arousal": arousal,
                    "moods": out.get("predicted_moods", [])}
        except Exception:
            return None


predictor = Music2EmoPredictor()

## 3. Datasets

Same dataset helpers as the Essentia notebook.

In [ ]:
df_deam,     deam_id, deam_val, deam_aro = setup_deam(DATA_DIR)
df_emomusic, em_id,   em_val,   em_aro   = setup_emomusic(DATA_DIR)
df_pmemo,    pm_id,   pm_val,   pm_aro   = setup_pmemo(DATA_DIR)
df_merge,    mg_id,   mg_val,   mg_aro   = setup_merge(DATA_DIR)

## 4. Evaluation

Adjust audio paths if your files are elsewhere (e.g. mounted Google Drive).

In [ ]:
_deam_audio   = next(iter(sorted((DATA_DIR / "deam").rglob("MEMD_audio"))), None)
DEAM_AUDIO    = _deam_audio if _deam_audio else DATA_DIR / "deam" / "MEMD_audio"
if _deam_audio:
    print(f"DEAM audio found at: {DEAM_AUDIO}")

EMOMUSIC_AUDIO = DATA_DIR / "emomusic" / "clips"

_pmemo_chorus = next(iter(sorted((DATA_DIR / "pmemo").rglob("chorus"))), None)
PMEMO_AUDIO   = _pmemo_chorus if _pmemo_chorus else DATA_DIR / "pmemo" / "chorus"
if _pmemo_chorus:
    print(f"PMEmo audio found at: {PMEMO_AUDIO}")

_merge_audio  = next(iter(sorted((DATA_DIR / "merge").rglob("audio"))), None)
MERGE_AUDIO   = _merge_audio if _merge_audio else DATA_DIR / "merge" / "audio"
if _merge_audio:
    print(f"MERGE audio found at: {MERGE_AUDIO}")

MAX_TRACKS = 200

DATASETS = [
    ("DEAM",     df_deam,     DEAM_AUDIO,     deam_id, deam_val, deam_aro),
    ("EmoMusic", df_emomusic, EMOMUSIC_AUDIO, em_id,   em_val,   em_aro),
    ("PMEmo",    df_pmemo,    PMEMO_AUDIO,    pm_id,   pm_val,   pm_aro),
    ("MERGE",    df_merge,    MERGE_AUDIO,    mg_id,   mg_val,   mg_aro),
]

all_results = {}
for ds_name, df_a, audio_dir, id_col, val_col, aro_col in DATASETS:
    if df_a is None or not audio_dir.exists():
        print(f"Skipping {ds_name} — audio not found at {audio_dir}")
        continue
    all_results[ds_name] = run_evaluation(
        ds_name, MODEL_TAG, predictor.predict,
        audio_dir, df_a, id_col, val_col, aro_col, MAX_TRACKS,
    )

In [ ]:
for ds_name, df in all_results.items():
    print_metrics(df, ds_name)
    plot_scatter(df, f"music2emo — {ds_name}")

In [ ]:
if all_results:
    combined = pd.concat(all_results.values())
    out = f"{MODEL_TAG}_results.csv"
    combined.to_csv(out, index=False)
    print(f"Saved: {out}  ({len(combined)} rows)")
else:
    print("No results to save — check that audio directories exist.")

## 5. Qualitative Spot-checks

In [ ]:
download_spot_checks(SPOTCHECK_DIR)

In [ ]:
spot_rows = []
for track in SPOT_CHECKS:
    audio = SPOTCHECK_DIR / f"{track['title']}.mp3"
    if not audio.exists():
        print(f"  ✗ {track['title']} — not found, skipping")
        continue
    pred = predictor.predict(audio)
    spot_rows.append({
        "title":       track["title"].replace("_", " "),
        "exp_valence": track["exp_valence"],
        "exp_arousal": track["exp_arousal"],
        "valence":     pred["valence"] if pred else float("nan"),
        "arousal":     pred["arousal"] if pred else float("nan"),
        "moods":       ", ".join((pred.get("moods") or [])[:4]) if pred else "",
    })
    name = track["title"].replace("_", " ").title()
    print(f"\n{name}")
    print(f"  Expected:  v={track['exp_valence']:.2f}  a={track['exp_arousal']:.2f}")
    if pred:
        moods = ", ".join((pred.get("moods") or [])[:4])
        print(f"  Predicted: v={pred['valence']:.2f}  a={pred['arousal']:.2f}  moods=[{moods}]")
    else:
        print("  Predicted: FAILED")

spot_df = pd.DataFrame(spot_rows)
print(f"\nCompleted {len(spot_df)} spot-checks.")

In [ ]:
if not spot_df.empty:
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(spot_df)))
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color="#ccc", lw=0.8, ls="--"); ax.axvline(0.5, color="#ccc", lw=0.8, ls="--")
    ax.text(0.02, 0.98, "sad/energetic",   transform=ax.transAxes, va="top",    fontsize=8, color="#999")
    ax.text(0.98, 0.98, "happy/energetic",  transform=ax.transAxes, va="top",    ha="right", fontsize=8, color="#999")
    ax.text(0.02, 0.02, "sad/calm",         transform=ax.transAxes, va="bottom", fontsize=8, color="#999")
    ax.text(0.98, 0.02, "happy/calm",       transform=ax.transAxes, va="bottom", ha="right", fontsize=8, color="#999")
    for i, row in spot_df.iterrows():
        c = colors[i]
        ax.scatter(row["exp_valence"], row["exp_arousal"], marker="*", s=220, color=c, zorder=4)
        if not pd.isna(row["valence"]):
            ax.scatter(row["valence"], row["arousal"], marker="o", s=70,
                       color=c, edgecolors="black", lw=0.5, zorder=4)
            ax.annotate("", xy=(row["valence"], row["arousal"]),
                        xytext=(row["exp_valence"], row["exp_arousal"]),
                        arrowprops=dict(arrowstyle="->", color=c, lw=1.0))
        ax.annotate(row["title"], xy=(row["exp_valence"], row["exp_arousal"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=7, color=c)
    ax.set_xlabel("Valence (sad → happy)")
    ax.set_ylabel("Arousal (calm → energetic)")
    ax.set_title("music2emo — Spot-checks\n★ = expected  ● = predicted")
    plt.tight_layout()
    plt.savefig(f"{MODEL_TAG}_spotchecks.png", dpi=120, bbox_inches="tight")
    plt.show()

## 6. Runtime & Memory Profiling

In [ ]:
test_audio = next(
    (SPOTCHECK_DIR / f"{t['title']}.mp3" for t in SPOT_CHECKS
     if (SPOTCHECK_DIR / f"{t['title']}.mp3").exists()),
    None,
)

if test_audio is None:
    print("No audio for profiling — run Section 5 first.")
else:
    prof = profile_predictor(predictor.predict, test_audio, n=5)
    print(f"music2emo — profiling on {test_audio.name}:")
    print(f"  Mean: {prof['mean_s']:.2f} s/track")
    print(f"  Std:  {prof['std_s']:.3f} s")
    print(f"  Peak RAM: {prof['peak_mb']:.1f} MB  (Python heap; GPU memory not included)")

## 7. Summary & Cross-Model Comparison

The cross-model comparison requires `essentia_results.csv` from notebook 1.
Place it in the same working directory, or update the path below.

In [ ]:
print("=" * 60)
print("MUSIC2EMO — BENCHMARK SUMMARY")
print("=" * 60)

print("\n── Dataset metrics ──")
if all_results:
    combined_m2e = pd.concat(all_results.values())
    print_metrics(combined_m2e, "all datasets")
else:
    print("  (no datasets evaluated)")

print("\n── Spot-checks ──")
if not spot_df.empty:
    print(spot_df[["title", "exp_valence", "exp_arousal", "valence", "arousal", "moods"]]
          .to_string(index=False, float_format="{:.2f}".format))
else:
    print("  (none run)")

print("\n── Runtime ──")
if "prof" in dir():
    print(f"  {prof['mean_s']:.2f} s/track  (peak RAM {prof['peak_mb']:.1f} MB)")
else:
    print("  (run Section 6 to profile)")

print("\n" + "=" * 60)

In [ ]:
# ── Cross-model comparison (requires essentia_results.csv from notebook 1) ───
essentia_csv = Path("essentia_results.csv")

if not essentia_csv.exists():
    print(f"⚠  {essentia_csv} not found.")
    print("   Run 01_essentia_benchmark.ipynb first and copy the CSV here.")
elif not all_results:
    print("No music2emo results to compare — run Section 4 first.")
else:
    df_essentia = pd.read_csv(essentia_csv)
    combined_all = pd.concat([df_essentia, pd.concat(all_results.values())])
    cross_dataset_comparison(combined_all)